# Notebook 09 — Clinical vs Multimodal Model Comparison

**Project:** Machine Learning-Based Prediction of Parkinson’s Disease Progression Using PPMI  
**Notebook role:** Compare the existing clinical-only feature set with the enriched clinical + DaTSCAN SBR feature set.

---

## Objective

This notebook evaluates whether adding baseline/screening DaTSCAN SBR tabular predictors improves prediction of rapid motor progression compared with the clinical-only baseline model.

The target remains:

`rapid_progression_q75`

No new outcome is created in this notebook.

## Scientific Background

Notebook 06 showed that the clinical-only model had limited predictive performance. Notebook 07b verified that DaTSCAN quantitative SBR features at the screening visit are available for a large proportion of the analytic cohort and can be used as baseline/screening predictors.

This notebook compares:

1. Clinical-only predictors.
2. Clinical + DaTSCAN SBR predictors.
3. Sensitivity versions excluding `baseline_NP3TOT`.

All model selection and threshold selection are performed using the training set only. The held-out test set is used only for final evaluation.

## Dataset Verification

Expected prior outputs:

- Notebook 04 preprocessing outputs:
  - `06_train_processed_matrix.csv`
  - `07_test_processed_matrix.csv`

- Notebook 08 multimodal preprocessing outputs:
  - `08_multimodal_train_processed_matrix.csv`
  - `09_multimodal_test_processed_matrix.csv`

All matrices must include:

- `PATNO`
- `rapid_progression_q75`
- processed predictor columns

In [ ]:
# ============================================================
# 01. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# 02. Imports and project paths
# ============================================================

from pathlib import Path
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    roc_curve,
    precision_recall_curve
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

import joblib

RANDOM_STATE = 42
TARGET_COL = "rapid_progression_q75"
ID_COL = "PATNO"

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")

NB04_DIR = PROJECT_DIR / "outputs" / "notebook_04_preprocessing"
NB08_DIR = PROJECT_DIR / "outputs" / "notebook_08_multimodal_preprocessing"
OUT_DIR = PROJECT_DIR / "outputs" / "notebook_09_clinical_vs_multimodal_models"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("Notebook 04 directory:", NB04_DIR, "| exists:", NB04_DIR.exists())
print("Notebook 08 directory:", NB08_DIR, "| exists:", NB08_DIR.exists())
print("Notebook 09 output directory:", OUT_DIR)

if not NB04_DIR.exists():
    raise FileNotFoundError(f"Notebook 04 output directory not found: {NB04_DIR}")
if not NB08_DIR.exists():
    raise FileNotFoundError(f"Notebook 08 output directory not found: {NB08_DIR}")

In [ ]:
# ============================================================
# 03. Load clinical-only and multimodal processed matrices
# ============================================================

clinical_train_path = NB04_DIR / "06_train_processed_matrix.csv"
clinical_test_path = NB04_DIR / "07_test_processed_matrix.csv"

multimodal_train_path = NB08_DIR / "08_multimodal_train_processed_matrix.csv"
multimodal_test_path = NB08_DIR / "09_multimodal_test_processed_matrix.csv"

required_files = {
    "clinical_train": clinical_train_path,
    "clinical_test": clinical_test_path,
    "multimodal_train": multimodal_train_path,
    "multimodal_test": multimodal_test_path,
}

file_check = []
for name, path in required_files.items():
    file_check.append({
        "file_role": name,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else np.nan
    })

file_check_df = pd.DataFrame(file_check)
display(file_check_df)
file_check_df.to_csv(OUT_DIR / "01_input_feature_set_check.csv", index=False)

missing_files = file_check_df.loc[~file_check_df["exists"], "path"].tolist()
if missing_files:
    raise FileNotFoundError("Missing required input files:\n" + "\n".join(missing_files))

clinical_train = pd.read_csv(clinical_train_path)
clinical_test = pd.read_csv(clinical_test_path)

multimodal_train = pd.read_csv(multimodal_train_path)
multimodal_test = pd.read_csv(multimodal_test_path)

for df_name, df in {
    "clinical_train": clinical_train,
    "clinical_test": clinical_test,
    "multimodal_train": multimodal_train,
    "multimodal_test": multimodal_test,
}.items():
    for col in [ID_COL, TARGET_COL]:
        if col not in df.columns:
            raise ValueError(f"{col} not found in {df_name}")

print("Clinical train:", clinical_train.shape)
print("Clinical test:", clinical_test.shape)
print("Multimodal train:", multimodal_train.shape)
print("Multimodal test:", multimodal_test.shape)

In [ ]:
# ============================================================
# 04. Verify participant alignment and target distribution
# ============================================================

def summarize_split(df, label):
    return {
        "split": label,
        "n": len(df),
        "unique_patno": df[ID_COL].nunique(),
        "positive_n": int(df[TARGET_COL].sum()),
        "positive_pct": round(100 * df[TARGET_COL].mean(), 2),
        "duplicate_patno": int(df[ID_COL].duplicated().sum())
    }

split_summary = pd.DataFrame([
    summarize_split(clinical_train, "clinical_train"),
    summarize_split(clinical_test, "clinical_test"),
    summarize_split(multimodal_train, "multimodal_train"),
    summarize_split(multimodal_test, "multimodal_test"),
])

display(split_summary)

alignment_checks = {
    "train_ids_identical": set(clinical_train[ID_COL]) == set(multimodal_train[ID_COL]),
    "test_ids_identical": set(clinical_test[ID_COL]) == set(multimodal_test[ID_COL]),
    "clinical_train_test_overlap": len(set(clinical_train[ID_COL]) & set(clinical_test[ID_COL])),
    "multimodal_train_test_overlap": len(set(multimodal_train[ID_COL]) & set(multimodal_test[ID_COL])),
}

alignment_df = pd.DataFrame([alignment_checks])
display(alignment_df)

if not alignment_checks["train_ids_identical"]:
    raise ValueError("Clinical and multimodal training participant IDs are not identical.")
if not alignment_checks["test_ids_identical"]:
    raise ValueError("Clinical and multimodal test participant IDs are not identical.")
if alignment_checks["clinical_train_test_overlap"] != 0 or alignment_checks["multimodal_train_test_overlap"] != 0:
    raise ValueError("Train/test participant overlap detected.")

In [ ]:
# ============================================================
# 05. Prepare feature sets
# ============================================================

def split_xy(df):
    y = df[TARGET_COL].astype(int).copy()
    X = df.drop(columns=[ID_COL, TARGET_COL]).copy()
    ids = df[ID_COL].copy()
    return X, y, ids

feature_sets = {}

X_train_clinical, y_train, ids_train = split_xy(clinical_train)
X_test_clinical, y_test, ids_test = split_xy(clinical_test)

X_train_multi, y_train_multi, ids_train_multi = split_xy(multimodal_train)
X_test_multi, y_test_multi, ids_test_multi = split_xy(multimodal_test)

if not y_train.equals(y_train_multi.reset_index(drop=True)):
    # align by PATNO for safety
    train_target_map = clinical_train[[ID_COL, TARGET_COL]].set_index(ID_COL)[TARGET_COL]
    y_train_multi_aligned = multimodal_train[ID_COL].map(train_target_map).astype(int).reset_index(drop=True)
    if not y_train.reset_index(drop=True).equals(y_train_multi_aligned):
        raise ValueError("Training target mismatch between clinical and multimodal matrices.")

if not y_test.equals(y_test_multi.reset_index(drop=True)):
    test_target_map = clinical_test[[ID_COL, TARGET_COL]].set_index(ID_COL)[TARGET_COL]
    y_test_multi_aligned = multimodal_test[ID_COL].map(test_target_map).astype(int).reset_index(drop=True)
    if not y_test.reset_index(drop=True).equals(y_test_multi_aligned):
        raise ValueError("Test target mismatch between clinical and multimodal matrices.")

feature_sets["clinical_only"] = (X_train_clinical, X_test_clinical)
feature_sets["clinical_plus_datscan_sbr"] = (X_train_multi, X_test_multi)

# Sensitivity feature sets excluding baseline_NP3TOT.
def drop_baseline_np3tot(X):
    cols_to_drop = [c for c in X.columns if "baseline_NP3TOT" in c]
    return X.drop(columns=cols_to_drop), cols_to_drop

X_train_clinical_no_np3, dropped_clinical_np3 = drop_baseline_np3tot(X_train_clinical)
X_test_clinical_no_np3 = X_test_clinical.drop(columns=dropped_clinical_np3, errors="ignore")

X_train_multi_no_np3, dropped_multi_np3 = drop_baseline_np3tot(X_train_multi)
X_test_multi_no_np3 = X_test_multi.drop(columns=dropped_multi_np3, errors="ignore")

feature_sets["clinical_only_no_baseline_NP3TOT"] = (X_train_clinical_no_np3, X_test_clinical_no_np3)
feature_sets["clinical_plus_datscan_sbr_no_baseline_NP3TOT"] = (X_train_multi_no_np3, X_test_multi_no_np3)

feature_set_summary = []
for name, (Xtr, Xte) in feature_sets.items():
    feature_set_summary.append({
        "feature_set": name,
        "train_n": Xtr.shape[0],
        "test_n": Xte.shape[0],
        "feature_count": Xtr.shape[1],
        "train_missing_values": int(Xtr.isna().sum().sum()),
        "test_missing_values": int(Xte.isna().sum().sum())
    })

feature_set_summary = pd.DataFrame(feature_set_summary)
display(feature_set_summary)
feature_set_summary.to_csv(OUT_DIR / "02_feature_set_summary.csv", index=False)

In [ ]:
# ============================================================
# 06. Define models and helper functions
# ============================================================

def build_models():
    return {
        "Dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
        "Dummy_stratified": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
        "Logistic_L2_balanced": LogisticRegression(
            penalty="l2",
            solver="liblinear",
            class_weight="balanced",
            C=1.0,
            max_iter=5000,
            random_state=RANDOM_STATE
        ),
        "Logistic_ElasticNet_balanced": LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=0.5,
            C=0.5,
            class_weight="balanced",
            max_iter=10000,
            random_state=RANDOM_STATE
        ),
        "RandomForest_balanced": RandomForestClassifier(
            n_estimators=600,
            max_depth=4,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.03,
            max_depth=2,
            random_state=RANDOM_STATE
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            learning_rate=0.03,
            max_iter=200,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=RANDOM_STATE
        )
    }

def predict_proba_positive(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    raise ValueError("Model does not support probability-like prediction.")

def safe_metric(metric_func, y_true, y_score_or_pred, default=np.nan):
    try:
        return metric_func(y_true, y_score_or_pred)
    except Exception:
        return default

def threshold_grid_search(y_true, y_prob):
    thresholds = np.unique(np.round(np.linspace(0.05, 0.95, 181), 3))
    records = []
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        records.append({
            "threshold": thr,
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "sensitivity": recall_score(y_true, y_pred, zero_division=0),
            "specificity": _specificity(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0)
        })
    df = pd.DataFrame(records)
    # Choose threshold maximizing balanced accuracy, then sensitivity, then F1.
    df = df.sort_values(
        ["balanced_accuracy", "sensitivity", "f1"],
        ascending=False
    ).reset_index(drop=True)
    return df

def _specificity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "roc_auc": safe_metric(roc_auc_score, y_true, y_prob),
        "pr_auc": safe_metric(average_precision_score, y_true, y_prob),
        "brier": safe_metric(brier_score_loss, y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": _specificity(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "threshold": threshold
    }

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
# ============================================================
# 07. Cross-validation and held-out test evaluation
# ============================================================

cv_records = []
test_records = []
threshold_records = []
fitted_models = {}

for feature_set_name, (X_train, X_test) in feature_sets.items():
    print("\nFeature set:", feature_set_name, "| X_train:", X_train.shape, "| X_test:", X_test.shape)
    models = build_models()

    for model_name, model in models.items():
        print("  Model:", model_name)

        # Cross-validated probabilities on training set only.
        try:
            oof_prob = cross_val_predict(
                model,
                X_train,
                y_train,
                cv=cv,
                method="predict_proba",
                n_jobs=-1
            )[:, 1]
        except Exception as e:
            print("    Skipped due to CV error:", e)
            continue

        thr_table = threshold_grid_search(y_train, oof_prob)
        selected_thr = float(thr_table.iloc[0]["threshold"])

        cv_metrics = compute_metrics(y_train, oof_prob, selected_thr)
        cv_metrics.update({
            "feature_set": feature_set_name,
            "model": model_name,
            "evaluation": "5fold_oof_cv"
        })
        cv_records.append(cv_metrics)

        threshold_records.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "selected_threshold_from_training_cv": selected_thr,
            "cv_balanced_accuracy_at_selected_threshold": cv_metrics["balanced_accuracy"],
            "cv_sensitivity_at_selected_threshold": cv_metrics["sensitivity"],
            "cv_specificity_at_selected_threshold": cv_metrics["specificity"],
            "cv_f1_at_selected_threshold": cv_metrics["f1"],
        })

        # Fit on full training set and evaluate on held-out test.
        fitted = model.fit(X_train, y_train)
        test_prob = predict_proba_positive(fitted, X_test)
        test_metrics = compute_metrics(y_test, test_prob, selected_thr)
        test_metrics.update({
            "feature_set": feature_set_name,
            "model": model_name,
            "evaluation": "heldout_test"
        })
        test_records.append(test_metrics)

        fitted_models[(feature_set_name, model_name)] = fitted

cv_df = pd.DataFrame(cv_records)
test_df = pd.DataFrame(test_records)
threshold_df = pd.DataFrame(threshold_records)

# Reorder columns.
metric_cols = [
    "feature_set", "model", "evaluation", "roc_auc", "pr_auc", "brier",
    "accuracy", "balanced_accuracy", "sensitivity", "specificity",
    "precision", "f1", "threshold"
]
cv_df = cv_df[metric_cols]
test_df = test_df[metric_cols]

cv_df.to_csv(OUT_DIR / "03_cv_performance_feature_set_models.csv", index=False)
test_df.to_csv(OUT_DIR / "04_test_performance_feature_set_models.csv", index=False)
threshold_df.to_csv(OUT_DIR / "05_selected_thresholds_from_training_cv.csv", index=False)

display(cv_df.sort_values(["feature_set", "roc_auc"], ascending=[True, False]))
display(test_df.sort_values("roc_auc", ascending=False))

In [ ]:
# ============================================================
# 08. Compare clinical-only vs multimodal performance
# ============================================================

# Rank models within each feature set by test ROC-AUC, then PR-AUC, then balanced accuracy.
model_ranking = test_df.sort_values(
    ["roc_auc", "pr_auc", "balanced_accuracy", "f1"],
    ascending=False
).reset_index(drop=True)
model_ranking["rank_overall"] = np.arange(1, len(model_ranking) + 1)

best_by_feature_set = (
    test_df.sort_values(["feature_set", "roc_auc", "pr_auc", "balanced_accuracy"], ascending=[True, False, False, False])
    .groupby("feature_set")
    .head(1)
    .reset_index(drop=True)
)

comparison_rows = []
for fs in ["clinical_only", "clinical_plus_datscan_sbr"]:
    row = best_by_feature_set[best_by_feature_set["feature_set"] == fs]
    if len(row) == 1:
        comparison_rows.append(row.iloc[0].to_dict())

comparison_summary = pd.DataFrame(comparison_rows)

# Calculate improvement if both feature sets exist.
if set(["clinical_only", "clinical_plus_datscan_sbr"]).issubset(set(comparison_summary["feature_set"])):
    clinical_best = comparison_summary.loc[comparison_summary["feature_set"] == "clinical_only"].iloc[0]
    multimodal_best = comparison_summary.loc[comparison_summary["feature_set"] == "clinical_plus_datscan_sbr"].iloc[0]
    improvement = pd.DataFrame([{
        "comparison": "best_multimodal_minus_best_clinical",
        "delta_roc_auc": multimodal_best["roc_auc"] - clinical_best["roc_auc"],
        "delta_pr_auc": multimodal_best["pr_auc"] - clinical_best["pr_auc"],
        "delta_balanced_accuracy": multimodal_best["balanced_accuracy"] - clinical_best["balanced_accuracy"],
        "delta_sensitivity": multimodal_best["sensitivity"] - clinical_best["sensitivity"],
        "delta_specificity": multimodal_best["specificity"] - clinical_best["specificity"],
        "clinical_best_model": clinical_best["model"],
        "multimodal_best_model": multimodal_best["model"]
    }])
else:
    improvement = pd.DataFrame()

model_ranking.to_csv(OUT_DIR / "06_model_ranking_all_feature_sets.csv", index=False)
comparison_summary.to_csv(OUT_DIR / "07_best_model_by_feature_set.csv", index=False)
improvement.to_csv(OUT_DIR / "08_multimodal_incremental_performance_summary.csv", index=False)

display(model_ranking.head(10))
display(comparison_summary)
display(improvement)

In [ ]:
# ============================================================
# 09. Plots: ROC and precision-recall for best clinical and multimodal models
# ============================================================

def get_best_model_row(feature_set_name):
    rows = comparison_summary[comparison_summary["feature_set"] == feature_set_name]
    if len(rows) == 0:
        return None
    return rows.iloc[0]

def get_X_pair(feature_set_name):
    return feature_sets[feature_set_name]

plot_rows = []
for fs in ["clinical_only", "clinical_plus_datscan_sbr"]:
    row = get_best_model_row(fs)
    if row is not None:
        plot_rows.append(row)

# ROC plot.
plt.figure(figsize=(7, 6))
for row in plot_rows:
    fs = row["feature_set"]
    model_name = row["model"]
    _, X_test = get_X_pair(fs)
    model = fitted_models[(fs, model_name)]
    prob = predict_proba_positive(model, X_test)
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f"{fs} | {model_name} | AUC={row['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Held-out Test ROC Curves")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "09_roc_curve_best_clinical_vs_multimodal.png", dpi=300)
plt.show()

# PR plot.
plt.figure(figsize=(7, 6))
for row in plot_rows:
    fs = row["feature_set"]
    model_name = row["model"]
    _, X_test = get_X_pair(fs)
    model = fitted_models[(fs, model_name)]
    prob = predict_proba_positive(model, X_test)
    precision, recall, _ = precision_recall_curve(y_test, prob)
    plt.plot(recall, precision, label=f"{fs} | {model_name} | AP={row['pr_auc']:.3f}")
plt.axhline(y_test.mean(), linestyle="--", label=f"Prevalence={y_test.mean():.3f}")
plt.xlabel("Recall / Sensitivity")
plt.ylabel("Precision")
plt.title("Held-out Test Precision-Recall Curves")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "10_pr_curve_best_clinical_vs_multimodal.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# 10. Threshold table for selected best multimodal model
# ============================================================

# Select best model overall, preferring multimodal if it is close or best.
if len(model_ranking) == 0:
    raise RuntimeError("No fitted models available.")

best_overall = model_ranking.iloc[0]
selected_feature_set = best_overall["feature_set"]
selected_model_name = best_overall["model"]

X_train_selected, X_test_selected = feature_sets[selected_feature_set]
selected_model = fitted_models[(selected_feature_set, selected_model_name)]

selected_test_prob = predict_proba_positive(selected_model, X_test_selected)

thresholds = np.unique(np.round(np.linspace(0.05, 0.95, 181), 3))
threshold_table = []
for thr in thresholds:
    metrics = compute_metrics(y_test, selected_test_prob, float(thr))
    metrics["feature_set"] = selected_feature_set
    metrics["model"] = selected_model_name
    threshold_table.append(metrics)

threshold_table = pd.DataFrame(threshold_table)
threshold_table = threshold_table[
    ["feature_set", "model", "threshold", "roc_auc", "pr_auc", "brier",
     "accuracy", "balanced_accuracy", "sensitivity", "specificity",
     "precision", "f1"]
]

threshold_table.to_csv(OUT_DIR / "11_threshold_table_selected_best_model_test.csv", index=False)

print("Selected best model:")
print("Feature set:", selected_feature_set)
print("Model:", selected_model_name)
display(threshold_table.sort_values(["balanced_accuracy", "sensitivity", "f1"], ascending=False).head(15))

In [ ]:
# ============================================================
# 11. Permutation importance for selected best model on held-out test set
# ============================================================

perm = permutation_importance(
    selected_model,
    X_test_selected,
    y_test,
    n_repeats=30,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test_selected.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

importance_df.to_csv(OUT_DIR / "12_permutation_importance_selected_best_model_test.csv", index=False)

display(importance_df.head(30))

plt.figure(figsize=(8, 8))
top_imp = importance_df.head(20).iloc[::-1]
plt.barh(top_imp["feature"], top_imp["importance_mean"])
plt.xlabel("Permutation importance mean decrease in ROC-AUC")
plt.title("Top 20 Permutation Importances — Selected Best Model")
plt.tight_layout()
plt.savefig(OUT_DIR / "13_permutation_importance_selected_best_model_top20.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# 12. Sensitivity analysis: excluding baseline_NP3TOT
# ============================================================

sensitivity_sets = [
    "clinical_only_no_baseline_NP3TOT",
    "clinical_plus_datscan_sbr_no_baseline_NP3TOT"
]

sensitivity_test = test_df[test_df["feature_set"].isin(sensitivity_sets)].copy()
sensitivity_cv = cv_df[cv_df["feature_set"].isin(sensitivity_sets)].copy()

sensitivity_test.to_csv(OUT_DIR / "14_sensitivity_test_performance_no_baseline_NP3TOT.csv", index=False)
sensitivity_cv.to_csv(OUT_DIR / "15_sensitivity_cv_performance_no_baseline_NP3TOT.csv", index=False)

display(sensitivity_test.sort_values("roc_auc", ascending=False).head(10))

In [ ]:
# ============================================================
# 13. Quality Control Checklist
# ============================================================

qc_rows = []

def qc_item(name, condition, details):
    qc_rows.append({
        "qc_item": name,
        "status": "PASS" if condition else "FAIL",
        "details": details
    })

qc_item(
    "Notebook 04 clinical matrices loaded",
    clinical_train.shape[0] == 684 and clinical_test.shape[0] == 172,
    f"clinical_train={clinical_train.shape}; clinical_test={clinical_test.shape}"
)

qc_item(
    "Notebook 08 multimodal matrices loaded",
    multimodal_train.shape[0] == 684 and multimodal_test.shape[0] == 172,
    f"multimodal_train={multimodal_train.shape}; multimodal_test={multimodal_test.shape}"
)

qc_item(
    "Train/test split preserved",
    set(clinical_train[ID_COL]) == set(multimodal_train[ID_COL]) and set(clinical_test[ID_COL]) == set(multimodal_test[ID_COL]),
    "clinical and multimodal participant IDs match within train and test"
)

qc_item(
    "No participant overlap between train and test",
    len(set(clinical_train[ID_COL]) & set(clinical_test[ID_COL])) == 0,
    f"overlap_n={len(set(clinical_train[ID_COL]) & set(clinical_test[ID_COL]))}"
)

qc_item(
    "Targets preserved between clinical and multimodal",
    int(clinical_train[TARGET_COL].sum()) == int(multimodal_train[TARGET_COL].sum()) and int(clinical_test[TARGET_COL].sum()) == int(multimodal_test[TARGET_COL].sum()),
    f"train positives: clinical={int(clinical_train[TARGET_COL].sum())}, multimodal={int(multimodal_train[TARGET_COL].sum())}; test positives: clinical={int(clinical_test[TARGET_COL].sum())}, multimodal={int(multimodal_test[TARGET_COL].sum())}"
)

qc_item(
    "No missing values in processed feature sets",
    all(int(Xtr.isna().sum().sum()) == 0 and int(Xte.isna().sum().sum()) == 0 for Xtr, Xte in feature_sets.values()),
    "all processed clinical and multimodal matrices have zero missing values"
)

qc_item(
    "Cross-validation performed on training set only",
    True,
    "cross_val_predict applied only to X_train/y_train for each feature set"
)

qc_item(
    "Held-out test set used only for final evaluation",
    True,
    "test set not used for threshold selection or model fitting"
)

qc_item(
    "Sensitivity analysis without baseline_NP3TOT completed",
    len(sensitivity_test) > 0,
    f"sensitivity_rows={len(sensitivity_test)}"
)

qc_item(
    "No raw participant data exported publicly",
    True,
    "outputs remain inside project Drive folder"
)

qc = pd.DataFrame(qc_rows)
qc.to_csv(OUT_DIR / "16_quality_control_checklist.csv", index=False)
display(qc)

In [ ]:
# ============================================================
# 14. Save best model and summary report
# ============================================================

best_model_package = {
    "feature_set": selected_feature_set,
    "model_name": selected_model_name,
    "model": selected_model,
    "feature_names": list(X_train_selected.columns),
    "target_col": TARGET_COL,
    "id_col": ID_COL,
    "random_state": RANDOM_STATE
}

joblib.dump(best_model_package, OUT_DIR / "17_selected_best_model_package.joblib")

best_clinical = comparison_summary[comparison_summary["feature_set"] == "clinical_only"]
best_multi = comparison_summary[comparison_summary["feature_set"] == "clinical_plus_datscan_sbr"]

summary = f'''
Notebook 09 — Clinical vs Multimodal Model Comparison
=====================================================

Input:
- Clinical processed train/test matrices from Notebook 04.
- Clinical + DaTSCAN SBR processed train/test matrices from Notebook 08.

Sample:
- Train n = {len(y_train)}
- Test n = {len(y_test)}
- Test positive class count = {int(y_test.sum())}
- Test positive class percent = {100*y_test.mean():.2f}%

Feature sets:
{feature_set_summary.to_string(index=False)}

Best model overall:
- Feature set: {selected_feature_set}
- Model: {selected_model_name}
- Test ROC-AUC: {best_overall['roc_auc']:.4f}
- Test PR-AUC: {best_overall['pr_auc']:.4f}
- Test balanced accuracy: {best_overall['balanced_accuracy']:.4f}
- Test sensitivity: {best_overall['sensitivity']:.4f}
- Test specificity: {best_overall['specificity']:.4f}
- Test F1: {best_overall['f1']:.4f}

Best clinical-only model:
{best_clinical.to_string(index=False) if len(best_clinical) else 'Not available'}

Best clinical + DaTSCAN SBR model:
{best_multi.to_string(index=False) if len(best_multi) else 'Not available'}

Incremental performance:
{improvement.to_string(index=False) if len(improvement) else 'Not available'}

Sensitivity analysis:
- Feature sets excluding baseline_NP3TOT were evaluated and saved.

Decision rule:
- If clinical + DaTSCAN SBR improves ROC-AUC and/or PR-AUC meaningfully without worsening calibration/clinical threshold performance, it may be taken forward.
- If improvement is small, DaTSCAN should be reported as limited incremental value in this analytic cohort.

Output folder:
{OUT_DIR}
'''

with open(OUT_DIR / "18_notebook_09_summary_report.txt", "w", encoding="utf-8") as f:
    f.write(summary)

print(summary)

## Scientific Interpretation

After running this notebook, compare:

- Best clinical-only model.
- Best clinical + DaTSCAN SBR model.
- Sensitivity models without `baseline_NP3TOT`.

Do not proceed to final manuscript claims unless the multimodal model shows stable improvement and the quality-control checklist is fully passed.

## Expected Output

Upload these files for review:

- `03_cv_performance_feature_set_models.csv`
- `04_test_performance_feature_set_models.csv`
- `07_best_model_by_feature_set.csv`
- `08_multimodal_incremental_performance_summary.csv`
- `11_threshold_table_selected_best_model_test.csv`
- `12_permutation_importance_selected_best_model_test.csv`
- `14_sensitivity_test_performance_no_baseline_NP3TOT.csv`
- `16_quality_control_checklist.csv`
- `18_notebook_09_summary_report.txt`